In [ ]:
from physics.simulation import mcfm, msq
from physics.hzz import zz4l
from physics.hstar import c6

import numpy as np
import json
import hist
import matplotlib, matplotlib.pyplot as plt


In [ ]:
matplotlib.use("pgf")
matplotlib.rcParams.update({
    "pgf.texsystem": "lualatex",
    "text.usetex": True,
    "pgf.rcfonts": False,
    "pgf.preamble": "", 
})

In [2]:
COMPONENT = msq.Component.SBI

with open('data/xsec.json') as f:
    xsec = json.load(f)

xsec = {
    msq.Component.SBI : xsec['gg4l_sbi'],
    msq.Component.SIG : xsec['gg4l_sig'],
    msq.Component.INT : xsec['gg4l_int'],
    msq.Component.BKG : xsec['gg4l_bkg']
}

filenames = {
    msq.Component.SBI : 'data/ggZZ2e2m_sbi/events.csv',
    msq.Component.SIG : 'data/ggZZ2e2m_sig/events.csv',
    msq.Component.INT : 'data/ggZZ2e2m_int/events.csv',
    msq.Component.BKG : 'data/ggZZ2e2m_bkg/events.csv'
}

component_names = {
    msq.Component.SBI : 'SBI',
    msq.Component.SIG : 'SIG',
    msq.Component.INT : 'INT',
    msq.Component.BKG : 'BKG'
}

In [3]:
events = mcfm.from_csv(cross_section=xsec[COMPONENT], file_path=filenames[COMPONENT])
events = zz4l.analyze(events)

In [4]:
xobs = events.kinematics['4l_mass'].to_numpy()
nxbins = 41
xmin, xmax = 180, 1000.0
xbins = np.linspace(xmin, xmax, nxbins + 1)
xcenters = 0.5 * (xbins[:-1] + xbins[1:])
xwidths = np.diff(xbins)

In [5]:
color_sm = 'black'
line_sm = '--'

In [ ]:
c6_vals = [20, -20]
c6_colors = ['red', 'blue']
c6_lines = ['-', '-']

mod_c6 = c6.Modifier(baseline=COMPONENT, events=events, c6_values=[-20,-5,0,5,20])
wt_c6, prob_c6 = mod_c6.modify(c6_vals)

In [8]:
h_sm = hist.Hist(hist.axis.Variable(xbins), storage=hist.storage.Weight())
h_sm.fill(xobs, weight=events.weights)

h_c6 = []
for i_c6, c6_val in enumerate(c6_vals):
    h = hist.Hist(hist.axis.Variable(xbins), storage=hist.storage.Weight())
    h.fill(xobs, weight=wt_c6[:,i_c6])
    h_c6.append(h)

In [9]:
lw  = 1.25

fig, (ax1, ax2) = plt.subplots(2,1, gridspec_kw={'height_ratios': [3,1]}, figsize=(5,5), sharex=True)

l_c6 = []
for i_c6, c6_val in enumerate(c6_vals):
    l_c6.append(ax1.stairs(h_c6[i_c6].values(), xbins, color=c6_colors[i_c6], linestyle=c6_lines[i_c6], linewidth=lw))
l_sm = ax1.stairs(h_sm.values(), xbins, color=color_sm, linestyle=line_sm, linewidth=lw)

l_sm.set_label('$\mathrm{SM}$')
for i_c6, c6_val in enumerate(c6_vals):
    sgn = '-' if c6_val < 0 else '+'
    l_c6[i_c6].set_label(f'$c_{{\phi}} = {sgn}{abs(c6_val):d}$')
ax1.legend(frameon=False)

for i_c6, c6_val in enumerate(c6_vals):
    ax2.stairs(h_c6[i_c6].values() / h_sm.values(), xbins, color=c6_colors[i_c6], linestyle=c6_lines[i_c6], linewidth=lw)
ax2.stairs(h_sm.values() / h_sm.values(), xbins, color=color_sm, linestyle=line_sm, linewidth=lw)

ax1.set_yscale('log')
ax1.set_ylabel('$\\mathrm{d}\sigma / \\mathrm{d} m_{ZZ}\\ \\mathrm{[fb / GeV]}$', loc='top')

ax2.set_ylabel('$\\mathrm{BSM} / \\mathrm{SM}$')
ax2.set_ylim(0.95,1.25)

ax2.set_xscale('linear')
ax2.set_xlim(xmin, xmax)
ax2.set_xlabel('$m_{ZZ}\\ \\mathrm[GeV]$', loc='right')
ax2.tick_params(top=True, labeltop=False)

fig.tight_layout()
fig.subplots_adjust(hspace=0)

fig.canvas.draw()  # update positions
ax1_pos, ax2_pos = ax1.get_position(), ax2.get_position()
ax2.set_position([ax1_pos.x0, ax2_pos.y0, ax1_pos.width, ax2_pos.height]) # align 2nd x-axis with 1st

plt.savefig('4l_mzz.pdf')
plt.close()